# Data Preprocessing - Sheffield Road Collision Dataset

This notebook covers the complete preprocessing pipeline for the "Filtered_Sheffield_Traffic_Data.csv". The pipeline is structured as follows:
1.

In [ ]:
# Install the relevant packages in the environment this notebook runs in
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn scipy pyyaml

---
# 1. Configuration and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def find_project_root(marker='config.yml'):
    current = Path().resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f'Could not find {marker} in any parent directory')

ROOT_DIR = find_project_root()
NOTEBOOKS_DIR = ROOT_DIR / 'notebooks'

with open(ROOT_DIR / 'config.yml') as f:
    pipeline_cfg = yaml.safe_load(f)['preprocessing']

with open(NOTEBOOKS_DIR / 'notebook-config.yml') as f:
    nb_cfg = yaml.safe_load(f)

NB_CONFIG = {
    'figsize_wide': nb_cfg['plotting']['figsize_wide'],
    'figsize_square': nb_cfg['plotting']['figsize_square'],
    'palette': nb_cfg['plotting']['palette'],
}

from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
from scipy import stats

sns.set_theme(style='whitegrid', palette=NB_CONFIG['palette'])
print(f'Project root: {ROOT_DIR}')
print('Pipeline config loaded from: config.yml')
print('Notebook config loaded from: notebook-config.yml')


---
# 2. Load and Initial Exploration

In [ ]:
df_raw = pd.read_csv(pipeline_cfg['raw_data_path'], encoding='utf-8-sig')

print(f'Shape: {df_raw.shape}')
print('Column dtypes:{df_raw.dtypes}')
print('First 3 rows:')
df_raw.head(3)

df_raw.describe()

In [ ]:
# True NaN missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_simmary = pd.DataFrame({'missing_count': missing, "missing_%": missing_pct})
print('Columns with NaN missing values:')
missing_simmary[missing_simmary['missing_count'] > 0]

In [ ]:
# STATS19 sentinel -1 values (encoded as "unknown/not applicable" in the raw data)
numeric_cols = df_raw.select_dtypes(include="number").columns
sentinel_counts = {
    col: (df_raw[col] == pipeline_cfg['sentinel_value']).sum()
    for col in numeric_cols
}
sentinel_df = pd.DataFrame.from_dict(sentinel_counts, orient="index", columns=["sentinel_-1_count"])
print('Columns containing sentinel -1 values:')
sentinel_df[sentinel_df['sentinel_-1_count'] > 0].sort_values('sentinel_-1_count', ascending=False)

In [ ]:
# Duplicate rows and duplicate collision_index entries
print(f'Fully duplicate rows: {df_raw.duplicated().sum()}')
print(f'Duplicate collision_index values: {df_raw["collision_index"].duplicated().sum()}')
print(f'\nYear range: {df_raw["collision_year"].min()} - {df_raw["collision_year"].max()}')

---
# 3. Visualize Raw Data

In [ ]:
fig, ax = plt.subplots(figsize=nb_cfg['figsize_wide'])
sns.heatmap(
    df_raw.isnull(),
    yticklabels=False,
    cbar=True,
    cmap='viridis',
    ax=ax
)
ax.set_title("Missing Value Heatmap (Raw Data)", fontsize=14, fontweight="bold")
ax.set_xlabel("Columns")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Sentinel -1 distribution across key columns
sentinel_plot_data = sentinel_df[sentinel_df['sentinel_-1_count'] > 0].sort_values('sentinel_-1_count', ascending=False)

fig, ax = plt.subplots(figsize=nb_cfg['figsize_wide'])
bars = ax.bar(sentinel_plot_data.index, sentinel_plot_data['sentinel_-1_count'], color=sns.color_palette(nb_cfg['palette']))
ax.set_title('Count of Sentinel (-1) Values per Column (Raw Data)', fontsize=14, fontweight='bold')
ax.set_xlabel('Column')
ax.set_ylabel('Count of -1 Values')
plt.xticks(rotation=45, ha='right')

for bar in bars:
    ax.annotate(
        f'{int(bar.get_height())}',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords='offset points',
        ha='center', fontsize=8
    )
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key target related columns
cols_to_plot = ['collision_severity', 'number_of_vehicles', 'number_of_casualties', 'speed_limit']
fig, axes = plt.subplots(1, len(cols_to_plot), figsize=nb_cfg['figsize_wide'])

for ax, col in zip(axes, cols_to_plot):
    df_raw[col].value_counts().sort_index().plot(kind='bar', ax=ax, color=sns.color_palette(nb_cfg['palette'])[0])
    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)

fig.suptitle('Key Column Distributions - Before Cleaning', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Collision count over years
fig, ax = plt.subplots(figsize=(12, 4))
df_raw['collision_year'].value_counts().sort_index().plot(kind='line', ax=ax, marker='o', color='steelblue')
ax.set_title('Collision Frequency by Year (Raw Data)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Collisions')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot of continuous cols to spot outliers visually
fig, axes = plt.subplots(1, len(pipeline_cfg['outlier_cols']), figsize=nb_cfg['figsize_wide'])

for ax, col in zip(axes, pipeline_cfg['outlier_cols']):
    ax.boxplot(df_raw[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xticks([])

fig.suptitle('Outlier Overview - Before Cleaning (Box Plots)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# 4. Drop Irrelvant/Redundant Columns

- Justification for each drop is defined in config.yml

In [ ]:
df = df_raw.copy()

# Drop only columns that actually exist in the dataframe (defensive)
cols_to_drop = [c for c in pipeline_cfg['drop_cols'] if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f'Columns dropped: {cols_to_drop}')
print(f'Shape after drop: {df.shape}')

---
# 5. Handle Sentinel Values

- In the STATS19 dataset, `-1` encodes "unknown" or "not applicable" so must be  treated as missing data.

In [ ]:
# Replace -1 with NaN in categorical colums, then mode-impute
for col in pipeline_cfg['sentinel_categorical_cols']:
    if col not in df.columns:
        continue
    n_sentinel = (df[col] == pipeline_cfg['sentinel_value']).sum()
    if n_sentinel > 0:
        df[col] = df[col].replace(pipeline_cfg['sentinel_value'], np.nan)
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: replaced {n_sentinel} sentinel values → mode imputed with {mode_val}')

# For sentinel_keep_as_category, recode -1 as a valid category integer
for col in pipeline_cfg['sentinel_keep_as_category']:
    if col not in df.columns:
        continue
    n_sentinel = (df[col] == pipeline_cfg['sentinel_value']).sum()
    print(f'{col}: keeping {n_sentinel} sentinel -1 values as "unknown" category')

---
# 6. Handle True Missing Values (NaN)

- After sentinel treatment, remaining NaNs are in spatial coordinate columns. KNN imputation is used to estimate these from neighbouring records, which is more principled than mean/median for geospatial data.

In [ ]:
# KNN Imputation for spatial coordinates
knn_cols = [c for c in pipeline_cfg['knn_impute_cols'] if c in df.columns]

print('Missing before KNN imputation:')
for col in knn_cols:
    print(f'  {col}: {df[col].isnull().sum()}')

knn_imputer = KNNImputer(n_neighbors=pipeline_cfg['knn_neighbours'])
df[knn_cols] = knn_imputer.fit_transform(df[knn_cols])

print('\nMissing after KNN imputation:')
for col in knn_cols:
    print(f'  {col}: {df[col].isnull().sum()}')

In [ ]:
# Drop longitude/latitude (replaced by easting/northing after imputation)
# longitude/latitude have 4240 NaNs and are redundant once we have easting/northing
for col in ['longitude', 'latitude']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

print('Remaining missing values after all imputation:')
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else 'None - dataset is complete.')

---
# 7. Fix Data Types

In [ ]:
df['date'] = pd.to_datetime(df['date'], dayfirst = True, errors="coerce")

df['hour_of_day'] = pd.to_datetime(df['time'], format='%H:%M', errors='coerce').dt.hour
df.drop(columns=['time'], inplace=True)

df['month'] = df['date'].dt.month
df.drop(columns=['date'], inplace=True)  # raw date not needed after feature extraction

int_cols = [
    'collision_severity', 'day_of_week', 'road_type', 'speed_limit',
    'junction_detail', 'junction_control', 'second_road_class',
    'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area',
    'first_road_class', 'collision_injury_based',
]
for col in int_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

print('Updated dtypes:')
print(df.dtypes)



---
# 8. Duplicate Detection and Precision

- The raw data has 609 duplicate `collision_index` values. These are not fully duplicate rows — they represent re-coded records across dataset editions (STATS19 data can have the same incident indexed differently in different years of release). We retain the most recently updated record.

In [ ]:
n_before = len(df)

df.sort_values('collision_year', ascending=False, inplace=True)
df.drop_duplicates(subset='collision_index', keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Rows before deduplication: {n_before}')
print(f'Rows after deduplication:  {len(df)}')
print(f'Rows removed: {n_before - len(df)}')

---
# 9. Outlier Detection and Treatment

- IQR based detection is used. Outliers are Winsorized rather than dropped, preserving record count and avoiding loss of (rare but) genuine events.

In [ ]:
def winsorize_iqr(series, multiplier=1.5):
    """Cap values outside IQR whiskers. Returns capped series and bounds."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 - multiplier * iqr
    return series.clip(lower=lower, upper=upper), lower, upper

outlier_cols = [c for c in pipeline_cfg['outlier_cols'] if c in df.columns]

# Capture pre-winsorisation state for comparison plot
df_pre_winsor = df[outlier_cols].copy()

for col in outlier_cols:
    n_outliers_before = (
        (df[col] < df[col].quantile(0.25) - pipeline_cfg['iqr_multiplier'] * (df[col].quantile(0.75) - df[col].quantile(0.25))) |
        (df[col] > df[col].quantile(0.75) + pipeline_cfg['iqr_multiplier'] * (df[col].quantile(0.75) - df[col].quantile(0.25)))
    ).sum()
    df[col], lower, upper = winsorize_iqr(df[col], pipeline_cfg['iqr_multiplier'])
    print(f'{col}: {n_outliers_before} outlier(s) Winsorized => bounds [{lower:.2f}, {upper:.2f}]')

In [ ]:
# Before vs After Winsorization box plots
fig, axes = plt.subplots(2, len(outlier_cols), figsize=(14, 7))

for i, col in enumerate(outlier_cols):
    axes[0, i].boxplot(df_pre_winsor[col], patch_artist=True, boxprops=dict(facecolor='salmon'))
    axes[0, i].set_title(f'{col}\n(Before)', fontsize=9)
    axes[0, i].set_xticks([])

    axes[1, i].boxplot(df[col], patch_artist=True, boxprops=dict(facecolor='lightgreen'))
    axes[1, i].set_title(f'{col}\n(After)', fontsize=9)
    axes[1, i].set_xticks([])

fig.suptitle('Outlier Treatment — Before vs After Winsorization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 10. Feature Engineering

- New features derived from existing columns to improve ML signal

In [ ]:
df['is_weekend'] = df['day_of_week'].isin([1, 7]).astype(int)

def hour_to_period(hour):
    if 6 <= hour < 12:
        return 0   # morning
    elif 12 <= hour < 17:
        return 1   # afternoon
    elif 17 <= hour < 21:
        return 2   # evening
    else:
        return 3   # night

df['time_period'] = df['hour_of_day'].apply(hour_to_period)

df['multi_vehicle'] = (df['number_of_vehicles'] > 1).astype(int)

print('New features added: is_weekend, time_period, multi_vehicle')
df[['is_weekend', 'time_period', 'multi_vehicle', 'hour_of_day']].head()

---
# 11. Scaling and Normalization

- Min-Max scaling is applied to continous numerical columns, bringing all values into [0, 1].

In [ ]:
scale_cols = [c for c in pipeline_cfg['scale_cols'] if c in df.columns]

# Capture pre-scale distributions
df_pre_scale = df[scale_cols].copy()

scaler = MinMaxScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print('Scaled columns (now in [0, 1]):')
df[scale_cols].describe().loc[['min', 'max']]

In [ ]:
# Before vs After scaling distributions
fig, axes = plt.subplots(2, len(scale_cols), figsize=(16, 6))

for i, col in enumerate(scale_cols):
    axes[0, i].hist(df_pre_scale[col], bins=30, color='salmon', edgecolor='white')
    axes[0, i].set_title(f'{col}\n(Before)', fontsize=8)

    axes[1, i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[1, i].set_title(f'{col}\n(After)', fontsize=8)

fig.suptitle('Min-Max Scaling — Before vs After Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 12. Post Cleaning Validation and Summary Visualizations

In [ ]:
# Final missing value check
final_missing = df.isnull().sum().sum()
print(f'Total missing values in cleaned dataset: {final_missing}')
print(f'Final shape: {df.shape}')
print(f'Columns retained: {list(df.columns)}')

In [ ]:
# Cleaned missing value heatmap
fig, ax = plt.subplots(figsize=nb_cfg['figsize_wide'])
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis', ax=ax)
ax.set_title('Missing Value Heatmap (Cleaned Data)', fontsize=14, fontweight='bold')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of cleaned numeric features
numeric_clean = df.select_dtypes(include='number').drop(columns=['collision_index'], errors='ignore')

fig, ax = plt.subplots(figsize=(14, 11))
corr = numeric_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=False, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=ax
)
ax.set_title('Correlation Matrix — Cleaned Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Collision severity class distribution (target variable check)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

severity_before = df_raw['collision_severity'].value_counts().sort_index()
severity_after  = df['collision_severity'].value_counts().sort_index()

severity_before.plot(kind='bar', ax=axes[0], color='salmon', edgecolor='white')
axes[0].set_title('Collision Severity — Before Cleaning', fontweight='bold')
axes[0].set_xlabel('Severity (1=Fatal, 2=Serious, 3=Slight)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

severity_after.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Collision Severity — After Cleaning', fontweight='bold')
axes[1].set_xlabel('Severity (1=Fatal, 2=Serious, 3=Slight)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Target Variable Distribution Before vs After Cleaning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: raw vs cleaned
summary = pd.DataFrame({
    'Metric': [
        'Total rows',
        'Total columns',
        'NaN missing values',
        'Sentinel -1 values (key cols)',
        'Duplicate collision_index',
    ],
    'Raw': [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        sum((df_raw[c] == -1).sum() for c in pipeline_cfg['sentinel_categorical_cols'] if c in df_raw.columns),
        df_raw['collision_index'].duplicated().sum(),
    ],
    'Cleaned': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        0,
        df['collision_index'].duplicated().sum(),
    ]
})
summary

In [ ]:
df.to_csv(pipeline_cfg['output_path'], index=False)
print(f'Cleaned dataset saved to: {pipeline_cfg["output_path"]}')